# **Lista de Exercícios: Aula 6 - Curso de Python FEA.dev**


**Instruções:**

- Resolva cada exercício na célula de código logo abaixo do enunciado.

- Caso encontre algum erro no material ou tenha dúvidas, entre em contato com os membros de Tech para que possamos ajudá-lo.

- Se estiver com dificuldade para realizar os exercícios, reveja as aulas institucionais, pesquise suas dúvidas em fóruns técnicos ou busque outros vídeos complementares. Consultar documentações e pesquisar ativamente faz parte da rotina dentro da entidade.

- Evite o uso de Inteligência Artificial para resolver os exercícios. O objetivo desta lista é que você desenvolva raciocínio lógico e autonomia na resolução de problemas.

**Sugestões:**

- Antes de começar, é recomendado ler o **Zen of Python** (execute a célula abaixo). São os princípios que guiam a linguagem e vão te dar uma noção do que a comunidade considera um bom código.

- Tente desenvolver os nomes de variáveis e os comentários dentro de cada célula em inglês. A maior parte do conteúdo disponível da linguagem se encontra em inglês, e caso queira compartilhar algum projeto com a comunidade externa, isso facilita que outras pessoas entendam diretamente seu código.

**Sobre esta lista:**

- Nesta lista você vai trabalhar como um **cientista de dados**: carregar uma base de dados real, limpá-la, explorá-la, criar análises estatísticas e gerar visualizações.

- A base de dados utilizada é o **National Fossil Carbon Emissions 2024 v1.0** (Global Carbon Project), que contém emissões de CO₂ fóssil por país desde 1850 até 2023.

- Os exercícios são **progressivos**: cada um constrói sobre o anterior. Ao final, você terá uma análise completa e profissional da base de dados.

In [ ]:
import this

---
## Tópico 0 — Setup e Carregamento dos Dados

Antes de iniciar os exercícios, precisamos importar as bibliotecas necessárias e carregar a base de dados. Execute as células abaixo.

In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

---
## Tópico 1 — Carregamento e Exploração Inicial da Base de Dados

**Exercício 1 (Prático)** — A base de dados `National.Fossil.Carbon.Emissions.2024v1.0.xlsx` possui múltiplas abas (sheets). Vamos trabalhar com a aba `'Territorial Emissions'`, que contém as emissões territoriais de CO₂ fóssil por país.

Porém, essa planilha possui **cabeçalhos descritivos nas primeiras linhas** que não são dados. Observe que:
- As linhas 0 a 8 (índice 0-based) contêm metadados e notas explicativas.
- A linha 9 contém a unidade de medida (`MtC/yr`).
- A linha 10 contém os nomes dos países em MAIÚSCULO.
- A linha 11 contém os nomes dos países em formato legível.
- A partir da linha 12, começam os dados numéricos (ano e emissões).

**a)** Use `pd.read_excel()` com o parâmetro `header=None` para carregar a aba `'Territorial Emissions'` sem interpretar nenhuma linha como cabeçalho. Armazene o resultado em `df_raw`.

**b)** Exiba o `shape` do DataFrame para verificar quantas linhas e colunas existem.

**c)** Use `.iloc` para exibir as 15 primeiras linhas e as 5 primeiras colunas (`df_raw.iloc[:15, :5]`). Observe a estrutura e identifique onde estão os metadados e onde começam os dados.

In [ ]:
# a) Load the raw data without header
df_raw = pd.read_excel(
    'National.Fossil.Carbon.Emissions.2024v1.0.xlsx',
    sheet_name='Territorial Emissions',
    header=None
)

# b) Check shape
print(f"Shape do DataFrame bruto: {df_raw.shape}")

# c) Preview first rows and columns
df_raw.iloc[:15, :5]


> **Explicação:** Ao usar `header=None`, o pandas não tenta interpretar nenhuma linha como nome de coluna — todos os nomes ficam como `0, 1, 2, ...`. Isso nos permite inspecionar o arquivo cru e decidir manualmente quais linhas são metadados e quais são dados. O `shape` mostra `(188, 231)`, indicando 188 linhas e 231 colunas (1 de anos + 230 de países/regiões).

**Exercício 2 (Prático)** — Agora que você entendeu a estrutura, vamos limpar a base de dados. Precisamos:

1. Extrair os nomes dos países a partir da **linha 11** (índice 11) — ela contém os nomes formatados corretamente.
2. Renomear a coluna 0 (que contém os anos) para `'Year'`.
3. Remover todas as linhas de metadados (linhas 0 a 11), mantendo apenas os dados a partir da linha 12.

**a)** Extraia os nomes dos países da linha 11 usando `df_raw.iloc[11, 1:]` e converta para uma lista. Armazene em `country_names`.

**b)** Crie um novo DataFrame `df` a partir de `df_raw.iloc[12:]` (dados a partir da linha 12). Use `.reset_index(drop=True)` para resetar o índice.

**c)** Renomeie as colunas: a coluna 0 deve se chamar `'Year'` e as colunas 1 em diante devem usar os nomes de `country_names`. **Dica:** construa a lista completa de nomes com `['Year'] + country_names`.

**d)** Exiba as 5 primeiras linhas com `.head()` e verifique se os nomes estão corretos.

In [ ]:
# a) Extract country names from row 11
country_names = df_raw.iloc[11, 1:].tolist()

# b) Create df with data rows only
df = df_raw.iloc[12:].reset_index(drop=True)

# c) Rename columns
df.columns = ['Year'] + country_names

# d) Preview
print(f"Shape após limpeza: {df.shape}")
df.head()


> **Explicação:** O `iloc[11, 1:]` pega toda a linha 11 a partir da coluna 1 (excluindo a coluna 0 que contém `NaN` / unidade). O `iloc[12:]` descarta todas as 12 primeiras linhas de metadados. Ao atribuir `df.columns`, substituímos os nomes numéricos por nomes significativos. Agora temos um DataFrame limpo com `'Year'` na primeira coluna e cada país nas demais.

**Exercício 3 (Prático)** — Vamos verificar e corrigir os **tipos de dados** (tipagem) do nosso DataFrame.

**a)** Use `df.dtypes` para verificar o tipo de cada coluna. O que você observa? As colunas numéricas estão como `object`?

**b)** Converta a coluna `'Year'` para `int` usando `.astype(int)`. Em seguida, converta todas as demais colunas (países) para `float` usando `.apply(pd.to_numeric, errors='coerce')` — isso garante que valores não numéricos virem `NaN` em vez de causar erro.

**c)** Verifique novamente os tipos com `df.dtypes` para confirmar que `'Year'` é `int64` e os países são `float64`.

**d)** Exiba o `.info()` do DataFrame para ter uma visão geral, incluindo a contagem de valores não-nulos por coluna.

In [ ]:
# a) Check dtypes
print("Tipos antes da conversão:")
print(df.dtypes.value_counts())

# b) Convert Year to int and countries to float
df['Year'] = df['Year'].astype(int)

country_cols = df.columns[1:]
df[country_cols] = df[country_cols].apply(pd.to_numeric, errors='coerce')

# c) Verify dtypes
print("\nTipos após a conversão:")
print(df.dtypes.value_counts())

# d) Display info
print("\nInfo do DataFrame:")
df.info()


> **Explicação:** Como o Excel mistura metadados textuais com dados numéricos, o pandas carrega tudo como `object`. O `pd.to_numeric(errors='coerce')` é essencial: ele tenta converter para número e, se falhar, substitui por `NaN` — evitando que o programa quebre. O `.info()` mostra quantos valores não-nulos cada coluna possui, o que nos ajuda a identificar colunas com muitos dados faltantes.

---
## Tópico 2 — Tratamento de Valores Ausentes

**Exercício 4 (Prático)** — Bases de dados reais quase sempre possuem valores ausentes (`NaN`). Vamos diagnosticar e tratar os valores faltantes.

**a)** Use `df.isna().sum().sum()` para contar o total de valores ausentes em todo o DataFrame. Exiba o resultado.

**b)** Selecione apenas as colunas de **países** (excluindo `'Year'` e as colunas de regiões ao final: `'Non KP Annex B'`, `'OECD'`, `'Non-OECD'`, `'EU27'`, `'Africa'`, `'Asia'`, `'Central America'`, `'Europe'`, `'Middle East'`, `'North America'`, `'Oceania'`, `'South America'`, `'Bunkers'`, `'Statistical Difference'`, `'World'`). Armazene essas colunas em uma lista chamada `region_cols` e crie `pure_country_cols` como as colunas que **não** estão em `['Year'] + region_cols`.

**c)** Calcule o **total de NaN por país** usando `df[pure_country_cols].isna().sum()`. Ordene de forma decrescente com `.sort_values(ascending=False)` e exiba os **10 países com mais valores ausentes**.

**d)** Preencha os valores ausentes das colunas de países com o valor anterior usando `.ffill()`. Se o primeiro ano for nulo, use também `.bfill()` em seguida. Confirme que não há mais `NaN` nas colunas de países.

In [ ]:
# a) Total NaN count
total_nan = df.isna().sum().sum()
print(f"Total de valores ausentes: {total_nan}")

# b) Separate country columns from region columns
region_cols = [
    'Non KP Annex B', 'OECD', 'Non-OECD', 'EU27',
    'Africa', 'Asia', 'Central America', 'Europe',
    'Middle East', 'North America', 'Oceania', 'South America',
    'Bunkers', 'Statistical Difference', 'World'
]

pure_country_cols = [col for col in df.columns if col not in ['Year'] + region_cols]
print(f"Total de países: {len(pure_country_cols)}")

# c) Top 10 countries with most NaN
nan_by_country = df[pure_country_cols].isna().sum().sort_values(ascending=False)
print("\nTop 10 países com mais valores ausentes:")
print(nan_by_country.head(10))

# d) Fill NaN with previous values (ffill) and verify
df[pure_country_cols] = df[pure_country_cols].ffill().bfill()
print(f"\nNaN restantes nos países: {df[pure_country_cols].isna().sum().sum()}")


> **Explicação:** O `isna()` retorna um DataFrame de booleanos (`True` onde há `NaN`). O `.sum()` conta os `True` por coluna, e `.sum().sum()` dá o total geral. A separação entre países e regiões é importante porque as colunas de regiões são **agregações** (somas de países), não dados individuais. Usamos `ffill()` (forward fill) para repetir o último valor válido e `bfill()` (backward fill) para cobrir os casos onde os primeiros anos são nulos.

**Exercício 5 (Prático)** — Limpeza de Outliers

Muitas vezes, bases de dados contêm valores anômalos (outliers) causados por erros de digitação ou medição. Vamos criar uma regra simples para tratar isso: **se um valor for maior que o dobro do ano anterior, ele será considerado um outlier e substituído pelo valor do ano anterior**.

Para aplicar isso de forma eficiente no pandas, vamos usar as funções `.shift()` (para pegar o valor do ano anterior) e `np.where()` (para aplicar a regra condicionalmente).

**a)** Primeiro, importe a biblioteca numpy caso ainda não tenha feito: `import numpy as np`.

**b)** Crie um loop `for` que itere por todas as colunas de países (`pure_country_cols`).

**c)** Dentro do loop, para cada país (`col`), crie uma Series contendo os valores do ano anterior usando `df[col].shift(1)`. O primeiro valor (1850) será `NaN`, então preencha com 0 usando `.fillna(0)`. Chame essa Series de `prev_values`.

**d)** Use `np.where()` para aplicar a regra: se o valor atual (`df[col]`) for maior que `2 * prev_values` **E** o valor atual for maior que 0.1 (para evitar alarmes falsos com valores muito próximos de zero, como 0.001 para 0.003), substitua por `prev_values`. Caso contrário, mantenha o valor atual `df[col]`. Atualize a coluna `df[col]` com o resultado.

**e)** Verifique o resultado exibindo as emissões do Brasil dos primeiros 10 anos.

In [ ]:
# a) Import numpy (already done in setup)
import numpy as np

# b) Loop through countries
for col in pure_country_cols:
    # c) Get previous year's values
    prev_values = df[col].shift(1).fillna(0)
    
    # d) Apply outlier rule using np.where()
    # Condição: valor atual > 2 * valor anterior E valor atual > 0.1 (ignora saltos irrelevantes em números tiny)
    condicao_outlier = (df[col] > 2 * prev_values) & (df[col] > 0.1)
    
    df[col] = np.where(condicao_outlier, prev_values, df[col])

# e) Verify Brazil's data
print("Primeiros anos do Brasil após limpeza de outliers:")
print(df[['Year', 'Brazil']].head(10))


> **Explicação:** A limpeza de outliers é crucial em dados de séries temporais. Usar `.shift(1)` permite alinhar o valor do ano atual com o do ano anterior sem a necessidade de um loop lento linha-a-linha. A condição `& (df[col] > 0.1)` evita que passagens de 0 para 0.01 disparem a regra (já que 0.01 é infinito vezes maior que 0). Com o `np.where()`, a substituição é feita de forma rápida e vetorizada.

---
## Tópico 3 — Filtragem e Operações Booleanas

**Exercício 6 (Teórico)** — Explique com suas palavras:

**a)** Qual é a diferença entre os operadores `&`, `|` e `~` no pandas e os operadores `and`, `or` e `not` do Python puro? Por que não podemos usar `and`/`or` para filtrar DataFrames?

**b)** O que o método `.isin()` faz? Dê um exemplo de quando ele seria útil em vez de usar vários filtros com `|`.

**c)** Qual é a diferença entre `dropna()` e `fillna()`? Em que situação cada um seria mais adequado?

**Resposta sugerida:**

**a)** Os operadores `&`, `|` e `~` operam **elemento a elemento** sobre Series/arrays booleanos do pandas, enquanto `and`, `or` e `not` operam sobre valores escalares (um único `True`/`False`). O pandas não sabe como avaliar a "verdade" de uma Series inteira com `and`, gerando o erro `The truth value of a Series is ambiguous`. Além disso, no pandas sempre devemos usar parênteses ao redor de cada condição: `(df['A'] > 5) & (df['B'] < 10)`.

**b)** O `.isin(lista)` verifica se cada valor de uma coluna pertence a uma lista de valores. Exemplo: `df[df['Country'].isin(['Brazil', 'Argentina', 'Chile'])]` é mais limpo do que `(df['Country'] == 'Brazil') | (df['Country'] == 'Argentina') | (df['Country'] == 'Chile')`.

**c)** O `dropna()` **remove** linhas/colunas com valores ausentes, reduzindo o tamanho do DataFrame. É adequado quando os dados faltantes são poucos e não comprometem a análise. O `fillna()` **substitui** os `NaN` por um valor (0, média, mediana, etc.), mantendo o tamanho do DataFrame. É mais adequado quando os valores faltantes têm um significado contextual (como zero emissões).

**Exercício 7 (Prático)** — Vamos aplicar filtros e operações booleanas na base de dados para extrair informações relevantes.

Para facilitar as análises, primeiro vamos transformar o DataFrame do formato **wide** (cada país é uma coluna) para o formato **long** (cada linha é um par país-ano). Use o método `pd.melt()`.

**a)** Crie `df_countries` selecionando apenas as colunas `['Year'] + pure_country_cols`. Em seguida, use `pd.melt()` com `id_vars='Year'`, `var_name='Country'` e `value_name='Emissions_MtC'` para transformar o DataFrame. Armazene em `df_long`.

**b)** Converta as emissões de MtC (milhões de toneladas de carbono) para MtCO₂ (milhões de toneladas de CO₂) multiplicando por `3.664`. Crie uma nova coluna `'Emissions_MtCO2'`.

**c)** Filtre `df_long` para exibir apenas dados do **Brasil** no **século XXI** (ano ≥ 2000). Use os operadores `&` com parênteses.

**d)** Use `.isin()` para filtrar os **5 maiores emissores históricos**: `['China', 'USA', 'Russia', 'Germany', 'United Kingdom']`. Exiba apenas os dados do ano de **2023**.

In [ ]:
# a) Melt to long format
df_countries = df[['Year'] + pure_country_cols]
df_long = pd.melt(
    df_countries,
    id_vars='Year',
    var_name='Country',
    value_name='Emissions_MtC'
)
print(f"Shape do df_long: {df_long.shape}")
print(df_long.head())

# b) Convert MtC to MtCO2
df_long['Emissions_MtCO2'] = df_long['Emissions_MtC'] * 3.664

# c) Filter: Brazil in the 21st century
brazil_21st = df_long[(df_long['Country'] == 'Brazil') & (df_long['Year'] >= 2000)]
print(f"\nBrasil no século XXI ({len(brazil_21st)} registros):")
print(brazil_21st.tail(10))

# d) Filter: Top 5 emitters in 2023 using isin()
top5 = ['China', 'USA', 'Russia', 'Germany', 'United Kingdom']
top5_2023 = df_long[(df_long['Country'].isin(top5)) & (df_long['Year'] == 2023)]
print(f"\nTop 5 emissores em 2023:")
print(top5_2023.sort_values('Emissions_MtCO2', ascending=False).to_string(index=False))


> **Explicação:** O `pd.melt()` é uma operação de *unpivot* — transforma colunas em linhas. Isso é fundamental para análise de dados, pois o formato *long* permite usar `groupby()`, filtros e visualizações de forma mais natural. A conversão de MtC para MtCO₂ usa o fator `3.664` (1 tonelada de carbono = 3.664 toneladas de CO₂). Os filtros com `&` exigem parênteses ao redor de cada condição devido à precedência de operadores do Python.

---
## Tópico 4 — GroupBy e Agregação

**Exercício 8 (Prático)** — Vamos usar `groupby()` e `agg()` para responder perguntas analíticas sobre as emissões globais.

**a)** Agrupe `df_long` por `'Country'` e calcule a **soma total de emissões em MtCO₂** de cada país ao longo de toda a história (1850–2023). Ordene de forma decrescente e exiba os **15 maiores emissores históricos acumulados**.

**b)** Agrupe por `'Year'` e calcule a **soma global** de emissões (todos os países juntos) por ano. Armazene em `global_emissions`.

**c)** Use `.agg()` para calcular, por país, **múltiplas estatísticas simultaneamente**: `sum`, `mean`, `max` e `std` das emissões em MtCO₂. Exiba os resultados para os **5 maiores emissores**.

**d)** Agrupe por **duas colunas**: crie uma coluna `'Decade'` calculada como `(df_long['Year'] // 10) * 10` e agrupe por `['Decade', 'Country']`. Calcule a soma de emissões por década e por país. Exiba as emissões do **Brasil por década**.

In [ ]:
# a) Top 15 historical emitters (total)
total_by_country = df_long.groupby('Country')['Emissions_MtCO2'].sum().sort_values(ascending=False)
print("Top 15 maiores emissores históricos (MtCO₂ acumulado):")
print(total_by_country.head(15).to_string())

# b) Global emissions by year
global_emissions = df_long.groupby('Year')['Emissions_MtCO2'].sum().reset_index()
global_emissions.columns = ['Year', 'Total_MtCO2']
print(f"\nEmissões globais - últimos 5 anos:")
print(global_emissions.tail(5).to_string(index=False))

# c) Multiple statistics with agg()
top5_names = total_by_country.head(5).index.tolist()
stats = df_long[df_long['Country'].isin(top5_names)].groupby('Country')['Emissions_MtCO2'].agg(
    ['sum', 'mean', 'max', 'std']
).sort_values('sum', ascending=False)
print(f"\nEstatísticas dos 5 maiores emissores:")
print(stats.to_string())

# d) Emissions by decade for Brazil
df_long['Decade'] = (df_long['Year'] // 10) * 10
decade_country = df_long.groupby(['Decade', 'Country'])['Emissions_MtCO2'].sum().reset_index()
brazil_decades = decade_country[decade_country['Country'] == 'Brazil'].sort_values('Decade')
print(f"\nEmissões do Brasil por década (MtCO₂):")
print(brazil_decades.to_string(index=False))


> **Explicação:** O `groupby()` divide o DataFrame em grupos baseados nos valores de uma ou mais colunas. Após o agrupamento, aplicamos uma função de agregação (`sum`, `mean`, `max`, `std`). O `agg()` permite calcular várias estatísticas de uma vez, retornando um DataFrame com uma coluna por métrica. A criação da coluna `'Decade'` usa divisão inteira (`//`) — por exemplo, `2023 // 10 * 10 = 2020`.

---
## Tópico 5 — Análise Estatística e Temporal

**Exercício 9 (Prático)** — Vamos realizar uma análise estatística e temporal das emissões, utilizando métodos como `describe()`, `corr()`, `diff()` e `pct_change()`.

**a)** Selecione as emissões de `'Brazil'`, `'USA'`, `'China'`, `'India'` e `'Germany'` do DataFrame original `df` (formato wide). Armazene em `df_selected`. Use `.describe()` para exibir as estatísticas descritivas.

**b)** Calcule a **matriz de correlação** dessas 5 colunas usando `.corr()`. Quais pares de países possuem correlação mais forte? A correlação alta significa que os dois países aumentaram/diminuíram suas emissões de forma sincronizada ao longo da história.

**c)** Usando `df_selected`, aplique `.diff()` na coluna `'Brazil'` para calcular a **diferença absoluta** entre anos consecutivos. Exiba os 10 maiores aumentos anuais de emissão do Brasil.

**d)** Aplique `.pct_change()` na coluna `'Brazil'` para calcular a **variação percentual** entre anos. Em qual ano o Brasil teve o maior aumento percentual de emissões? E a maior queda?

In [ ]:
# a) Select countries and describe
selected_countries = ['Brazil', 'USA', 'China', 'India', 'Germany']
df_selected = df[['Year'] + selected_countries].copy()
df_selected = df_selected.set_index('Year')
print("Estatísticas descritivas (MtC):")
print(df_selected.describe().to_string())

# b) Correlation matrix
corr_matrix = df_selected.corr()
print("\nMatriz de Correlação:")
print(corr_matrix.round(3).to_string())

# c) diff() - absolute difference for Brazil
df_selected['Brazil_diff'] = df_selected['Brazil'].diff()
top_increases = df_selected['Brazil_diff'].sort_values(ascending=False).head(10)
print("\n10 maiores aumentos anuais de emissão do Brasil (MtC):")
print(top_increases.to_string())

# d) pct_change() - percentage change for Brazil
df_selected['Brazil_pct'] = df_selected['Brazil'].pct_change() * 100
max_increase_year = df_selected['Brazil_pct'].idxmax()
max_decrease_year = df_selected['Brazil_pct'].idxmin()
print(f"\nMaior aumento percentual: {max_increase_year} ({df_selected.loc[max_increase_year, 'Brazil_pct']:.1f}%)")
print(f"Maior queda percentual: {max_decrease_year} ({df_selected.loc[max_decrease_year, 'Brazil_pct']:.1f}%)")


> **Explicação:** O `describe()` fornece estatísticas resumidas (count, mean, std, min, quartis, max). A `corr()` calcula a correlação de Pearson entre pares de colunas — valores próximos de 1 indicam correlação positiva forte, e próximos de -1, correlação negativa forte. O `diff()` calcula `valor[t] - valor[t-1]`, útil para ver variações absolutas. O `pct_change()` calcula `(valor[t] - valor[t-1]) / valor[t-1]`, útil para variações relativas (percentuais).

**Exercício 10 (Prático)** — Vamos usar `np.where()` para criar categorias e aplicar o método `.replace()` para limpar dados textuais.

**a)** No `df_long`, use `np.where()` para criar uma coluna `'Emission_Level'` que classifique as emissões em MtCO₂:
- `'Very Low'` se ≤ 1
- `'Low'` se ≤ 10
- `'Medium'` se ≤ 100
- `'High'` se ≤ 1000
- `'Very High'` se > 1000

**Dica:** Use `np.where()` encadeado (aninhado).

**b)** Conte quantos registros existem em cada nível usando `.value_counts()`.

**c)** Use `.replace()` para substituir os nomes dos níveis de inglês para português: `'Very Low'` → `'Muito Baixo'`, `'Low'` → `'Baixo'`, etc.

**d)** Filtre apenas os registros do ano de **2023** que possuem nível `'Muito Alto'`. Quais países estão nessa categoria?

In [ ]:
# a) Create Emission_Level with np.where()
df_long['Emission_Level'] = np.where(
    df_long['Emissions_MtCO2'] <= 1, 'Very Low',
    np.where(
        df_long['Emissions_MtCO2'] <= 10, 'Low',
        np.where(
            df_long['Emissions_MtCO2'] <= 100, 'Medium',
            np.where(
                df_long['Emissions_MtCO2'] <= 1000, 'High',
                'Very High'
            )
        )
    )
)

# b) Count records per level
print("Contagem por nível de emissão:")
print(df_long['Emission_Level'].value_counts())

# c) Replace English labels with Portuguese
df_long['Emission_Level'] = df_long['Emission_Level'].replace({
    'Very Low': 'Muito Baixo',
    'Low': 'Baixo',
    'Medium': 'Médio',
    'High': 'Alto',
    'Very High': 'Muito Alto'
})
print("\nApós tradução:")
print(df_long['Emission_Level'].value_counts())

# d) Very High emitters in 2023
very_high_2023 = df_long[
    (df_long['Year'] == 2023) & (df_long['Emission_Level'] == 'Muito Alto')
]
print(f"\nPaíses com emissão 'Muito Alto' em 2023:")
print(very_high_2023[['Country', 'Emissions_MtCO2']].sort_values('Emissions_MtCO2', ascending=False).to_string(index=False))


> **Explicação:** O `np.where(condição, valor_true, valor_false)` funciona como um `if-else` vetorizado — muito mais rápido que um loop `for`. Ao aninhar múltiplos `np.where()`, criamos categorias múltiplas (como um `if-elif-else`). O `.replace()` com um dicionário substitui valores de forma eficiente, sendo útil para padronizar ou traduzir labels.

---
## Tópico 6 — Visualização com Matplotlib e Seaborn

**Exercício 11 (Prático)** — Vamos criar visualizações profissionais para comunicar os resultados da nossa análise.

**a)** Use `matplotlib` para criar um **gráfico de linhas** das emissões globais totais ao longo do tempo (use `global_emissions` do Exercício 8b). Configure:
- Título: `'Emissões Globais de CO₂ Fóssil (1850–2023)'`
- Eixo x: `'Ano'`
- Eixo y: `'Emissões (MtCO₂)'`
- Tamanho da figura: `(12, 6)`

**b)** Use `matplotlib` para criar um **gráfico de linhas** comparando as emissões de `'Brazil'`, `'USA'`, `'China'`, `'India'` e `'Germany'` ao longo do tempo (use o DataFrame `df` original, formato wide). Adicione legenda.

**c)** Use `seaborn` para criar um **heatmap** da matriz de correlação calculada no Exercício 9b. Use `annot=True` para exibir os valores e `cmap='coolwarm'` para a paleta de cores.

**d)** Use `matplotlib` para criar um **gráfico de barras horizontal** dos 10 maiores emissores históricos acumulados (use o resultado do Exercício 8a). Use `plt.barh()` e ordene do maior para o menor.

In [ ]:
# a) Global emissions line chart
fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(global_emissions['Year'], global_emissions['Total_MtCO2'], color='crimson', linewidth=2)
ax.set_title('Emissões Globais de CO₂ Fóssil (1850–2023)', fontsize=16, fontweight='bold')
ax.set_xlabel('Ano', fontsize=12)
ax.set_ylabel('Emissões (MtCO₂)', fontsize=12)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
# b) Comparison line chart
fig, ax = plt.subplots(figsize=(12, 6))
countries_to_plot = ['Brazil', 'USA', 'China', 'India', 'Germany']
for country in countries_to_plot:
    ax.plot(df['Year'], df[country], linewidth=2, label=country)

ax.set_title('Emissões de CO₂ por País (1850–2023)', fontsize=16, fontweight='bold')
ax.set_xlabel('Ano', fontsize=12)
ax.set_ylabel('Emissões (MtC)', fontsize=12)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
# c) Correlation heatmap with seaborn
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    corr_matrix,
    annot=True,
    cmap='coolwarm',
    fmt='.2f',
    linewidths=0.5,
    ax=ax
)
ax.set_title('Matriz de Correlação — Emissões de CO₂', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
# d) Top 10 bar chart
top10 = total_by_country.head(10)

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(top10.index[::-1], top10.values[::-1], color='steelblue', edgecolor='navy')
ax.set_xlabel('Emissões Acumuladas (MtCO₂)', fontsize=12)
ax.set_title('Top 10 Maiores Emissores Históricos de CO₂', fontsize=16, fontweight='bold')
ax.grid(True, axis='x', alpha=0.3)
plt.tight_layout()
plt.show()


> **Explicação:** O `matplotlib` oferece controle total sobre cada elemento do gráfico (título, eixos, cores, tamanho). A estrutura `fig, ax = plt.subplots()` cria uma figura e um eixo, permitindo customizações finas. O `seaborn` simplifica gráficos estatísticos como o heatmap — `annot=True` exibe valores numéricos e `cmap` define a paleta de cores. O `barh()` cria barras horizontais, ideais para comparar categorias com nomes longos.

**Exercício 12 (Prático)** — Vamos criar visualizações mais avançadas para completar nossa análise.

**a)** Use `matplotlib` para criar um gráfico de **área empilhada** (`ax.stackplot()`) mostrando a contribuição dos 5 maiores emissores ao longo do tempo. Configure cores distintas e legenda.

**b)** Use `seaborn` para criar um **histograma** (`sns.histplot()`) com KDE (Kernel Density Estimation) das emissões de 2023 de todos os países. Use `kde=True` e `bins=30`.

**c)** Crie um **scatter plot** (gráfico de dispersão) com `matplotlib` comparando as emissões de 2000 vs 2023 para cada país. Adicione uma linha diagonal (`y = x`) para identificar países que aumentaram (acima da linha) ou diminuíram (abaixo) suas emissões.

In [ ]:
# a) Stacked area plot
fig, ax = plt.subplots(figsize=(14, 7))
top5_names_plot = total_by_country.head(5).index.tolist()
years = df['Year'].values
emissions_data = [df[c].values * 3.664 for c in top5_names_plot]

ax.stackplot(years, *emissions_data, labels=top5_names_plot, alpha=0.8)
ax.set_title('Emissões de CO₂ — Top 5 Emissores (Área Empilhada)', fontsize=16, fontweight='bold')
ax.set_xlabel('Ano', fontsize=12)
ax.set_ylabel('Emissões (MtCO₂)', fontsize=12)
ax.legend(loc='upper left', fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
# b) Histogram with KDE
emissions_2023 = df_long[df_long['Year'] == 2023]['Emissions_MtCO2']

fig, ax = plt.subplots(figsize=(10, 6))
sns.histplot(emissions_2023, bins=30, kde=True, color='teal', ax=ax)
ax.set_title('Distribuição das Emissões de CO₂ por País (2023)', fontsize=16, fontweight='bold')
ax.set_xlabel('Emissões (MtCO₂)', fontsize=12)
ax.set_ylabel('Frequência', fontsize=12)
plt.tight_layout()
plt.show()


In [ ]:
# c) Scatter plot: 2000 vs 2023
emissions_2000 = df_long[df_long['Year'] == 2000][['Country', 'Emissions_MtCO2']].rename(
    columns={'Emissions_MtCO2': 'Em_2000'}
)
emissions_2023_df = df_long[df_long['Year'] == 2023][['Country', 'Emissions_MtCO2']].rename(
    columns={'Emissions_MtCO2': 'Em_2023'}
)
comparison = emissions_2000.merge(emissions_2023_df, on='Country')

fig, ax = plt.subplots(figsize=(10, 10))
ax.scatter(comparison['Em_2000'], comparison['Em_2023'], alpha=0.6, s=40, color='darkorange')

# Diagonal line (y = x)
max_val = max(comparison['Em_2000'].max(), comparison['Em_2023'].max())
ax.plot([0, max_val], [0, max_val], 'k--', alpha=0.5, label='Sem mudança (y = x)')

# Annotate outliers
for _, row in comparison.nlargest(5, 'Em_2023').iterrows():
    ax.annotate(row['Country'], (row['Em_2000'], row['Em_2023']), fontsize=9)

ax.set_title('Emissões 2000 vs 2023 — Quem subiu e quem desceu?', fontsize=16, fontweight='bold')
ax.set_xlabel('Emissões em 2000 (MtCO₂)', fontsize=12)
ax.set_ylabel('Emissões em 2023 (MtCO₂)', fontsize=12)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


> **Explicação:** O `stackplot` mostra a contribuição individual de cada grupo para o total, permitindo visualizar tanto a tendência agregada quanto a composição. O `histplot` com `kde=True` combina um histograma (barras) com uma estimativa de densidade (curva suave), ideal para entender a distribuição dos dados. O scatter plot com a diagonal `y = x` é uma técnica poderosa: pontos acima da linha indicam aumento e pontos abaixo indicam diminuição entre os dois períodos.

---
## Desafio Final

**Desafio** — Crie uma **análise completa da América do Sul** utilizando todos os conceitos aprendidos nesta lista.

Os países da América do Sul presentes na base são: `'Argentina'`, `'Bolivia'`, `'Brazil'`, `'Chile'`, `'Colombia'`, `'Ecuador'`, `'Guyana'`, `'Paraguay'`, `'Peru'`, `'Suriname'`, `'Trinidad and Tobago'`, `'Uruguay'`, `'Venezuela'`.

**a)** Filtre `df_long` para incluir apenas os países da América do Sul. Armazene em `df_sa`.

**b)** Use `groupby()` e `agg()` para calcular, por país, a soma total e a média de emissões em MtCO₂. Ordene pela soma.

**c)** Calcule a **participação percentual** de cada país no total da América do Sul no ano de **2023**.

**d)** Crie uma visualização com **4 subplots** (2x2) usando `fig, axes = plt.subplots(2, 2, figsize=(16, 12))`:

  1. **Gráfico de linhas**: emissões dos 5 maiores emissores da América do Sul ao longo do tempo.
  2. **Gráfico de barras**: emissões totais acumuladas por país da América do Sul.
  3. **Gráfico de pizza**: participação percentual em 2023.
  4. **Heatmap**: matriz de correlação entre os 5 maiores emissores da América do Sul.

**e)** Adicione um título geral com `fig.suptitle()` e use `plt.tight_layout()` para organizar o layout.

In [ ]:
south_america = [
    'Argentina', 'Bolivia', 'Brazil', 'Chile', 'Colombia',
    'Ecuador', 'Guyana', 'Paraguay', 'Peru', 'Suriname',
    'Trinidad and Tobago', 'Uruguay', 'Venezuela'
]

# a) Filter South America
df_sa = df_long[df_long['Country'].isin(south_america)].copy()
print(f"Registros da América do Sul: {len(df_sa)}")

# b) GroupBy + Agg
sa_stats = df_sa.groupby('Country')['Emissions_MtCO2'].agg(['sum', 'mean']).sort_values('sum', ascending=False)
print("\nEstatísticas por país (MtCO₂):")
print(sa_stats.round(2).to_string())

# c) Percentage share in 2023
sa_2023 = df_sa[df_sa['Year'] == 2023][['Country', 'Emissions_MtCO2']].copy()
total_sa_2023 = sa_2023['Emissions_MtCO2'].sum()
sa_2023['Pct'] = (sa_2023['Emissions_MtCO2'] / total_sa_2023 * 100).round(1)
sa_2023 = sa_2023.sort_values('Emissions_MtCO2', ascending=False)
print(f"\nParticipação em 2023 (total = {total_sa_2023:.1f} MtCO₂):")
print(sa_2023.to_string(index=False))

# d) 4 subplots
top5_sa = sa_stats.head(5).index.tolist()

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1 - Line chart
for country in top5_sa:
    data = df_sa[df_sa['Country'] == country]
    axes[0, 0].plot(data['Year'], data['Emissions_MtCO2'], linewidth=2, label=country)
axes[0, 0].set_title('Emissões ao Longo do Tempo — Top 5 América do Sul', fontweight='bold')
axes[0, 0].set_xlabel('Ano')
axes[0, 0].set_ylabel('Emissões (MtCO₂)')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# 2 - Bar chart
axes[0, 1].barh(sa_stats.index[::-1], sa_stats['sum'].values[::-1], color='teal', edgecolor='darkslategray')
axes[0, 1].set_title('Emissões Acumuladas — América do Sul', fontweight='bold')
axes[0, 1].set_xlabel('Emissões Acumuladas (MtCO₂)')
axes[0, 1].grid(True, axis='x', alpha=0.3)

# 3 - Pie chart
# Group small slices into 'Outros'
pie_data = sa_2023.set_index('Country')['Emissions_MtCO2']
threshold = pie_data.sum() * 0.03
main_countries = pie_data[pie_data >= threshold]
others = pie_data[pie_data < threshold].sum()
if others > 0:
    main_countries['Outros'] = others
axes[1, 0].pie(main_countries, labels=main_countries.index, autopct='%1.1f%%', startangle=90)
axes[1, 0].set_title('Participação nas Emissões — América do Sul (2023)', fontweight='bold')

# 4 - Heatmap correlation
pivot_sa = df_sa[df_sa['Country'].isin(top5_sa)].pivot_table(
    index='Year', columns='Country', values='Emissions_MtCO2'
)
corr_sa = pivot_sa.corr()
sns.heatmap(corr_sa, annot=True, cmap='coolwarm', fmt='.2f', ax=axes[1, 1], linewidths=0.5)
axes[1, 1].set_title('Correlação — Top 5 América do Sul', fontweight='bold')

# e) Layout
fig.suptitle('Análise de Emissões de CO₂ — América do Sul', fontsize=18, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


> **Explicação:** Este desafio integra todos os conceitos da lista: filtragem com `isin()`, agregação com `groupby()` + `agg()`, cálculo de percentuais, e visualização com `matplotlib` e `seaborn`. A técnica de `subplots(2, 2)` cria uma grade de 4 gráficos em uma única figura, permitindo uma visão holística dos dados. O gráfico de pizza agrupa países pequenos em 'Outros' para melhor legibilidade. O heatmap de correlação revela se os padrões de emissão dos países sul-americanos são sincronizados.